## Libro: §7.1–7.6 del Capítulo 7 — Variedades generales, métrica de Bures y Wasserstein

**Cuaderno:** `cap7_wasserstein_quantico.ipynb`  
**Capítulo:** 7 — Temas Avanzados  
**Reproducibilidad:** SEED determinístico = `setup_seed("cap7_wasserstein_quantico")` (ver `notebooks/utils.py`).

Tres extensiones más allá de las familias exponenciales: (i) métrica de Fisher en familias no-exponenciales (t-Student con ν = 5); (ii) métrica de Bures en el espacio de estados cuánticos puros (qubit en la esfera de Bloch); (iii) Optimal Transport: W_1 vs KL en distribuciones discretas univariadas.

In [ ]:
import sys
sys.path.insert(0, '.')
from utils import setup_seed
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

SEED = setup_seed("cap7_wasserstein_quantico")
rng = np.random.default_rng(SEED)
print(f"SEED determinístico aplicado: {SEED}")


### 7.1 Métrica de Fisher para t-Student (ν = 5)

La familia t-Student **no es exponencial**, pero la métrica de Fisher está
bien definida:

    g_μμ(θ) = E[(∂ log p / ∂μ)²].

Comparamos g_μμ entre Normal y t-Student. En las colas (|x − μ| grande),
la t-Student es más sensible al parámetro de localización que la Normal,
y la métrica varía con ν.

In [ ]:
from scipy.stats import t as student_t

def fisher_localization_tstudent(nu, x_grid):
    # I(mu) = E[(dL/dmu)^2] para distribucion t-Student (Monte Carlo).
    samples = student_t.rvs(df=nu, size=100_000, random_state=rng)
    score_samples = (nu + 1) * samples / (nu + samples ** 2)  # μ=0
    return np.mean(score_samples ** 2)

I_t5 = fisher_localization_tstudent(5, dummy_grid := np.linspace(-30, 30, 1000))
I_normal = 1.0
print(f"I_t-Student(ν=5) ≈ {I_t5:.4f}")
print(f"I_Normal         = {I_normal:.4f}")

# Variación con ν
nus = [2, 3, 5, 10, 30]
It = [fisher_localization_tstudent(n, dummy_grid) for n in nus]
print("ν vs I(ν):", list(zip(nus, [round(x, 4) for x in It])))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(nus, It, "o-", color="#0F766E", linewidth=2, label="$I_t(\\nu)$")
ax.axhline(I_normal, color="red", linestyle=":", label="$I_{Normal}=1$")
ax.set_xlabel("Grados de libertad ν"); ax.set_ylabel("$I_\\mu(\\nu)$")
ax.set_title("Métrica de localización de t-Student vs Normal")
ax.legend(); ax.grid(True, alpha=0.3)
ax.invert_xaxis()
plt.tight_layout(); plt.show()


### 7.2 Métrica de Bures sobre el espacio de qubits

Un qubit puro |ψ⟩ = cos(θ/2)|0⟩ + e^{iφ} sin(θ/2)|1⟩ vive en la esfera
de Bloch. La distancia de Bures reducida a estados puros es:

    d_B(|ψ⟩, |φ⟩) = arccos |⟨ψ|φ⟩|.

Geodésica exacta en CP¹ (proyectivización de amplitudes).

In [ ]:
def qubit_state(theta, phi):
    # Estado qubit puro: cos(theta/2)|0> + e^{i phi} sin(theta/2)|1>.
    return np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)

def bures_distance_pure(t1, p1, t2, p2):
    a1, b1 = qubit_state(t1, p1)
    a2, b2 = qubit_state(t2, p2)
    overlap = a1 * a2.conjugate() + b1 * b2.conjugate()   # ⟨ψ₁|ψ₂⟩
    return np.arccos(np.clip(np.abs(overlap), 0, 1))

d_same  = bures_distance_pure(0.5, 0.3, 0.5, 0.3)
d_ortho = bures_distance_pure(0, 0, np.pi, 0)              # |0⟩ vs |1⟩: π/2
d_45    = bures_distance_pure(0, 0, np.pi / 2, 0)          # |0⟩ vs |+⟩: π/4
print(f"d_B(|ψ⟩, |ψ⟩)     = {d_same:.4f}  (esperado 0)")
print(f"d_B(|0⟩, |1⟩)     = {d_ortho:.4f}  (esperado π/2 ≈ {np.pi/2:.4f})")
print(f"d_B(|0⟩, |+⟩)     = {d_45:.4f}  (esperado π/4 ≈ {np.pi/4:.4f})")

# Heatmap de d_B(|0⟩, |θ,φ⟩) sobre la esfera de Bloch
theta = np.linspace(0, np.pi, 100)
phi   = np.linspace(0, 2 * np.pi, 100)
T, P  = np.meshgrid(theta, phi)
D = np.empty_like(T)
for i in range(T.shape[0]):
    for j in range(T.shape[1]):
        D[i, j] = bures_distance_pure(0, 0, T[i, j], P[i, j])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(phi * 180 / np.pi, theta * 180 / np.pi, D,
                   cmap="viridis", shading="auto")
ax.set_xlabel("φ (grados)"); ax.set_ylabel("θ (grados)")
ax.set_title("$d_B$( |0⟩, |θ,φ⟩ ) sobre la esfera de Bloch")
fig.colorbar(im, ax=ax, label="$d_B$")
plt.tight_layout(); plt.show()


### 7.3 Optimal Transport: Wasserstein vs KL

Para distribuciones discretas univariadas ordenadas:

    W_1(p, q) = Σ_{i=1}^{k−1} |F_p(x_i) − F_q(x_i)| · Δx_i.

A diferencia de KL — que diverge si los soportes no se solapan —, W_1
**siempre está bien definida** y captura la geometría del espacio
muestral.

In [ ]:
def wasserstein_1d_monotone(p, q, x):
    # W_1(p, q) para distribuciones discretas univariadas ordenadas.
    Fp = np.cumsum(p); Fq = np.cumsum(q)
    dx = np.diff(x)
    return np.sum(np.abs(Fp[:-1] - Fq[:-1]) * dx)

def kl_divergence(p, q):
    p_s = p + 1e-12; q_s = q + 1e-12
    return np.sum(p_s * np.log(p_s / q_s))

# Caso (★★) del .tex: W_1(Bern(0.3), Bern(0.7))
x_bin = np.array([0.0, 1.0])
p_b   = np.array([0.3, 0.7])
q_b   = np.array([0.7, 0.3])
W1_bern = wasserstein_1d_monotone(p_b, q_b, x_bin)
KL_bern = kl_divergence(p_b, q_b)
print(f"W_1(Bern(0.3), Bern(0.7)) = {W1_bern:.4f}")
print(f"KL( Bern(0.3) || Bern(0.7)) = {KL_bern:.4f}")

# Distribuciones con soporte disjunto — KL requiere épsilon
x_disj = np.array([0.0, 1.0, 2.0, 3.0])
p_d    = np.array([0.5, 0.5, 0.0, 0.0])
q_d    = np.array([0.0, 0.0, 0.5, 0.5])
W1_disj = wasserstein_1d_monotone(p_d, q_d, x_disj)
KL_disj = kl_divergence(p_d, q_d)
print(f"\nSoporte disjunto {{0,1}} vs {{2,3}}:")
print(f"  W_1 = {W1_disj:.4f}  (geometría: distancia 3)")
print(f"  KL  = {KL_disj:.4f}  (con ε para evitar 0/0)")


In [ ]:
def gaussian_geodesic(mu0, sigma0, mu1, sigma1, t):
    # Geodesica lineal en coordenadas naturales (eta).
    eta0 = np.array([mu0 / sigma0**2, -1 / (2 * sigma0**2)])
    eta1 = np.array([mu1 / sigma1**2, -1 / (2 * sigma1**2)])
    eta_t = (1 - t) * eta0 + t * eta1
    sigma_t = np.sqrt(-1 / (2 * eta_t[1]))
    mu_t = eta_t[0] * sigma_t**2
    return mu_t, sigma_t

# Plot: dos paneles
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Panel A: KL vs W_1 al separar gaussianas
mus = np.linspace(0, 5, 50)
KLs, W1s = [], []
x = np.linspace(-10, 15, 1000)
for m in mus:
    p = np.exp(-0.5 * x ** 2) / np.sqrt(2 * np.pi)
    q = np.exp(-0.5 * (x - m) ** 2) / np.sqrt(2 * np.pi)
    dx = np.diff(x).mean()
    KLs.append(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))) * dx)
    W1s.append(wasserstein_1d_monotone(p / np.sum(p), q / np.sum(q), x))

axes[0].plot(mus, KLs, color="#888888", label="KL$(p||q)$")
axes[0].plot(mus, W1s, color="#0F766E", linewidth=2, label="$W_1(p,q)$")
axes[0].set_xlabel("μ_q"); axes[0].set_ylabel("Divergencia")
axes[0].set_title("KL vs $W_1$: gaussianas separándose")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Panel B: geodésica Gaussiana
ts = np.linspace(0, 1, 50)
mus_path   = [gaussian_geodesic(0, 1, 3, 2, t)[0] for t in ts]
sigmas_path = [gaussian_geodesic(0, 1, 3, 2, t)[1] for t in ts]
axes[1].plot(mus_path, sigmas_path, "o-", color="#0F766E")
axes[1].set_xlabel("μ"); axes[1].set_ylabel("σ")
axes[1].set_title("Geodésica Gaussiana  N(0,1) → N(3,2)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 7.4 Ejercicios resueltos del capítulo

**(★★)** *Wasserstein entre Bernoulli(0.3) y Bernoulli(0.7).*

p = (0.3, 0.7) en {0,1}, q = (0.7, 0.3). CDFs: F_p = (0.3, 1),
F_q = (0.7, 1). Diferencia: |0.3 − 0.7| = 0.4 en [0, 1) (longitud 1).
W_1 = 0.4. Hay que mover 0.4 de masa de la posición 1 a la 0.

**(★★)** *Métrica de Fisher de Cauchy.*

Cauchy(μ) tiene densidad p(x) = π⁻¹ / [1 + (x − μ)²].
Score: ∂ℓ/∂μ = 2 (x − μ) / [1 + (x − μ)²].
E[score²] = ½ — constante e independiente de μ.

In [ ]:
# Verificación numérica: Fisher Cauchy
def fisher_cauchy(n_samples=200_000):
    samples = stats.cauchy.rvs(size=n_samples, random_state=rng)
    score = 2 * samples / (1 + samples ** 2)
    return score.var()

I_c = fisher_cauchy()
print(f"I_Cauchy (Monte Carlo) ≈ {I_c:.4f}")
print(f"Valor exacto teórico = 1/2 = 0.5000")

# Verificación: chequear puntos intermedios de la geodésica
print("\nGeodésica N(0,1) → N(3,2):")
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    mu_t, sigma_t = gaussian_geodesic(0, 1, 3, 2, t)
    print(f"   t={t:.2f}: μ={mu_t:.3f}, σ={sigma_t:.3f}")


## ✅ `@ Verifica con:`

t-Student(ν=5): métrica ≈ valor Monte Carlo con 100k muestras.
Bures: d_B(|0⟩,|1⟩) = π/2 ≈ 1.571; d_B(|0⟩,|+⟩) = π/4 ≈ 0.785.
W_1(Bern(0.3), Bern(0.7)) = 0.4; KL ≈ 0.797.
Fisher Cauchy ≈ 0.5 (teórico).